####Data Reading


In [0]:
df=spark.read.format("parquet")\
    .load("abfss://bronze@datalakedatabricksram.dfs.core.windows.net/orders")


In [0]:
df.display()

In [0]:

from pyspark.sql.functions import *
df=df.withColumn("order_date",to_timestamp(col("order_date")))
df.display()

In [0]:
df=df.withColumn("year", year(col("order_date")))
df.display()

####Classes in OOP

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
class windows:

    def dense_rank(self,df):
        df_dense_rank=df.withColumn("flag",dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_dense_rank
    
    def rank(self,df):
        df_rank=df.withColumn("rank_flag",rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_rank
    
    def row_number(self,df):
        df_row_number=df.withColumn("row_number",row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_row_number

In [0]:
df_new=df
df_new.display()

In [0]:
obj=windows()
df_result=obj.dense_rank(df_new)
df_result.display()


####Data Writing

In [0]:
df.write.format('delta').mode('overwrite').save("abfss://silver@datalakedatabricksram.dfs.core.windows.net/orders")

In [0]:
%sql 
create table if not exists databricks_catalog.silver.orders_silver
using delta
location 'abfss://silver@datalakedatabricksram.dfs.core.windows.net/orders'